# Connecting Claude Agent SDK to an External MCP Server

Unlike in-process custom tools, external MCP servers run as separate processes and are connected via configuration, not Python function definitions. This episode connects to the official filesystem MCP server, scoped to one throwaway demo directory.


In [1]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent  # this notebook lives in episodes/, project root is one level up
DEMO_DIR = PROJECT_ROOT / "09_mcp_demo"

# Plain Python: create a small demo folder + file for the external MCP
# server to serve. Nothing here talks to Claude yet.
DEMO_DIR.mkdir(parents=True, exist_ok=True)
(DEMO_DIR / "demo.txt").write_text("This file is served by an external MCP server.\n")
print("Created:", list(DEMO_DIR.iterdir()))

Created: [PosixPath('/Users/yashjain/Developer/youtube/claude-agent-sdk-youtube/09_mcp_demo/demo.txt')]


In [3]:
from claude_agent_sdk import (
    ClaudeSDKClient,  # a client you can keep open and send several messages through
    ClaudeAgentOptions,  # settings object: model, system prompt, tools, etc.
    AssistantMessage,  # message type that holds Claude's actual reply
    ToolUseBlock,  # message piece that shows "Claude is calling a tool now"
    ResultMessage,  # the last message in the stream — carries the final answer plus stats (cost, duration, etc.)
)


async def use_filesystem_server() -> None:
    options = ClaudeAgentOptions(
        model="haiku",
        # mcp_servers: this time, instead of a Python object (like episodes 7-8),
        # we describe a SEPARATE PROCESS to launch and talk to.
        #   "filesystem" -> our label for this server
        #   command/args -> the shell command that starts it: an official,
        #                   ready-made MCP server that knows how to read files,
        #                   scoped to only the DEMO_DIR path we pass in
        mcp_servers={
            "filesystem": {
                "command": "npx",
                "args": ["-y", "@modelcontextprotocol/server-filesystem", str(DEMO_DIR)],
            }
        },
        # allowed_tools: pre-approve every tool this MCP server offers (the "*" wildcard)
        allowed_tools=["mcp__filesystem__*"],
        # disallowed_tools: turn OFF the SDK's built-in file tools, so Claude
        # is forced to use the external MCP server instead — proving it really
        # is going through the separate process, not cheating with Read/Write.
        disallowed_tools=["Bash", "Read", "Glob", "Write", "Edit"],
        # add_dirs: the CLI's own workspace boundary is separate from the MCP
        # server's allowed directory — this grants Claude access to DEMO_DIR too
        add_dirs=[str(DEMO_DIR)],
    )
    async with ClaudeSDKClient(options=options) as client:
        await client.query(
            f"List the files in the directory {DEMO_DIR}, then read demo.txt from that exact "
            "directory and tell me what it says."
            "Write another line which is 'By: Yash Jain'"
        )
        async for message in client.receive_response():
            if isinstance(message, AssistantMessage):
                for block in message.content:
                    if isinstance(block, ToolUseBlock):
                        print(f"[tool call] {block.name}({block.input})")
            elif isinstance(message, ResultMessage):
                print(f"\nResult: {message.result}")


await use_filesystem_server()

[tool call] ToolSearch({'query': 'select:mcp__filesystem__list_directory,mcp__filesystem__read_text_file,mcp__filesystem__write_file', 'max_results': 5})
[tool call] mcp__filesystem__list_directory({'path': '/Users/yashjain/Developer/youtube/claude-agent-sdk-youtube/09_mcp_demo'})
[tool call] mcp__filesystem__read_text_file({'path': '/Users/yashjain/Developer/youtube/claude-agent-sdk-youtube/09_mcp_demo/demo.txt'})
[tool call] mcp__filesystem__write_file({'path': '/Users/yashjain/Developer/youtube/claude-agent-sdk-youtube/09_mcp_demo/demo.txt', 'content': 'This file is served by an external MCP server.\nBy: Yash Jain'})

Result: Done! ✅ I've successfully added the line "By: Yash Jain" to the demo.txt file. The file now contains:
```
This file is served by an external MCP server.
By: Yash Jain
```


That tool call went through a separate `npx`-launched process — not a Python function in this notebook. That's the difference from Episode 7/8's in-process custom tool.

## `.mcp.json` vs `~/.claude.json` scoping

MCP servers can also be declared outside your code:

- A **project-level `.mcp.json`** at the repo root is checked into version control and shared with everyone working in that project.
- **`~/.claude.json`** holds **user-level** servers available across every project on your machine, regardless of which repo you're in.

Passing `mcp_servers=` directly in `ClaudeAgentOptions`, as this notebook does, overrides/supplements both — useful when a server's config needs to be generated at runtime (like the demo path above) rather than hardcoded in a file.


In [4]:
# Plain Python cleanup — remove the demo folder we created for this notebook.
import shutil

shutil.rmtree(DEMO_DIR)
print("Deleted:", DEMO_DIR)

Deleted: /Users/yashjain/Developer/youtube/claude-agent-sdk-youtube/09_mcp_demo
